# Lab 7: Running Experiments and Building Custom Architectures

**Before you start: go to Runtime → Change runtime type and select GPU.**

You have a baseline. Now what?

The naive approach is to change things one after another and hope something 
helps. This leads to wasted compute, contradictory results, and a report 
that cannot explain why anything worked. The disciplined approach is to 
treat each change as an experiment: one variable at a time, everything 
else held fixed, results recorded before moving on.

This lab also introduces **custom architectures** — how to go beyond 
simply swapping the final layer of a pretrained model, and build 
something tailored to your task.

**This lab covers:**
1. Hyperparameter search — finding a good learning rate and weight decay
2. Ablation studies — proving your design choices actually matter
3. Logging results so you can compare them later
4. Building custom architectures: custom heads, multi-output models, 
   encoder-decoder structures

**Session structure:**
- First 45 minutes: work through this notebook
- Remaining time: design and run your first real experiment on your 
  own project with TA support

**Goal for today:** every group leaves with a structured experiment 
plan and at least one result beyond the baseline.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np
import itertools, json, zipfile, gdown
from pathlib import Path
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES} | Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Shared training function — used throughout this notebook
def train_model(model, train_dl, val_dl, epochs=5, lr=1e-3, weight_decay=0.0,
                seed=42, verbose=True):
    """
    Train a model and return (best_val_acc, best_val_f1, history).
    seed ensures reproducibility across experiments.
    """
    torch.manual_seed(seed)
    model     = deepcopy(model).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )
    best_val_acc = 0.0
    best_state   = None
    history      = {'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        t_corr = t_tot = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_corr += (out.argmax(1) == lbls).sum().item()
            t_tot  += imgs.size(0)

        model.eval()
        v_corr = v_tot = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                v_corr += (out.argmax(1) == lbls).sum().item()
                v_tot  += imgs.size(0)
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())

        val_acc = v_corr / v_tot
        history['train_acc'].append(t_corr / t_tot)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = deepcopy(model.state_dict())

        if verbose:
            print(f'  Epoch {epoch}/{epochs}  '
                  f'train_acc={t_corr/t_tot:.3f}  val_acc={val_acc:.3f}')

    val_f1 = f1_score(all_labels, all_preds, average='macro')
    return best_val_acc, val_f1, history, best_state


def make_resnet18(num_classes):
    """Standard pretrained ResNet-18 with replaced head."""
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

print('Utilities defined.')

## 1. Hyperparameter search

The two hyperparameters that matter most for most projects are:
- **Learning rate** — the most impactful single choice
- **Weight decay** — controls regularisation strength

The right strategy is **coarse-to-fine search on a log scale**:
1. Run short experiments (5 epochs) over a wide range of values
2. Identify the promising region
3. Run longer experiments in that region

Always search on a **log scale** — the difference between lr=0.001 and 
lr=0.01 is much more important than the difference between lr=0.1 and lr=0.11.

In [ ]:
# Step 1: Coarse search over learning rates
# We use only 5 epochs and a subset of the data for speed
from torch.utils.data import Subset
import random

# Use 30% of training data for quick hyperparameter search
random.seed(42)
subset_idx   = random.sample(range(len(train_ds)), int(0.3 * len(train_ds)))
subset_train = Subset(train_ds, subset_idx)
subset_dl    = DataLoader(subset_train, batch_size=32, shuffle=True, num_workers=2)

lr_candidates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
lr_results    = []

print('Learning rate search (5 epochs, 30% of data):')
for lr in lr_candidates:
    model = make_resnet18(NUM_CLASSES)
    val_acc, val_f1, _, _ = train_model(
        model, subset_dl, val_dl, epochs=5, lr=lr, verbose=False
    )
    lr_results.append((lr, val_acc))
    print(f'  lr={lr:.0e}  val_acc={val_acc:.3f}')

# Plot
lrs, accs = zip(*lr_results)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lrs, accs, 'o-', color='steelblue', markersize=8)
ax.set_xlabel('Learning rate (log scale)')
ax.set_ylabel('Validation accuracy')
ax.set_title('Coarse learning rate search')
ax.grid(alpha=0.3)
best_lr = lrs[np.argmax(accs)]
ax.axvline(best_lr, color='red', linestyle='--', label=f'Best: {best_lr:.0e}')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Best learning rate from coarse search: {best_lr:.0e}')

In [ ]:
# Step 2: Fine search around the best learning rate
# Also search over weight decay
fine_lrs  = [best_lr / 3, best_lr, best_lr * 3]
wds       = [0.0, 1e-4, 1e-3]

fine_results = []
print('Fine search (lr x weight_decay grid, 5 epochs, 30% of data):')
print(f'  {'lr':<10} {'wd':<10} {'val_acc':<10}')
print('  ' + '-' * 32)

for lr, wd in itertools.product(fine_lrs, wds):
    model = make_resnet18(NUM_CLASSES)
    val_acc, _, _, _ = train_model(
        model, subset_dl, val_dl,
        epochs=5, lr=lr, weight_decay=wd, verbose=False
    )
    fine_results.append({'lr': lr, 'wd': wd, 'val_acc': val_acc})
    print(f'  {lr:<10.2e} {wd:<10.0e} {val_acc:<10.3f}')

# Find best combination
best_config = max(fine_results, key=lambda x: x['val_acc'])
print(f'\nBest config: lr={best_config["lr"]:.2e}, wd={best_config["wd"]:.0e}, '
      f'val_acc={best_config["val_acc"]:.3f}')

## 2. Ablation studies

A hyperparameter search asks: *what value works best?*
An ablation study asks: *does this component actually matter?*

Ablations are how you justify your design choices in your report. 
Without ablations, you cannot claim that any specific decision 
contributed to your result — it might have been something else.

The pattern is always the same: start with your full model, 
remove or change one component at a time, and measure the effect.

Here we ablate three components of our training setup:

In [ ]:
# Define the ablation conditions
# Each condition changes exactly one thing relative to the baseline

def make_transform(augment=True):
    """Create transforms with or without data augmentation."""
    steps = [transforms.Resize((224, 224))]
    if augment:
        steps += [
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    steps += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

# No augmentation dataset
train_ds_noaug = datasets.ImageFolder(data_dir / 'train',
                                       transform=make_transform(augment=False))
train_dl_noaug = DataLoader(train_ds_noaug, batch_size=32, shuffle=True, num_workers=2)

# Ablation conditions
ablations = [
    {
        'name':        'Full model (baseline)',
        'model':       make_resnet18(NUM_CLASSES),
        'train_dl':    train_dl,
        'lr':          best_config['lr'],
        'wd':          best_config['wd'],
    },
    {
        'name':        'No data augmentation',
        'model':       make_resnet18(NUM_CLASSES),
        'train_dl':    train_dl_noaug,   # only change
        'lr':          best_config['lr'],
        'wd':          best_config['wd'],
    },
    {
        'name':        'No weight decay',
        'model':       make_resnet18(NUM_CLASSES),
        'train_dl':    train_dl,
        'lr':          best_config['lr'],
        'wd':          0.0,              # only change
    },
    {
        'name':        'Frozen backbone',
        'model':       (lambda m: [
            [setattr(p, 'requires_grad', False)
             for n, p in m.named_parameters() if 'fc' not in n],
            m
        ][-1])(make_resnet18(NUM_CLASSES)),  # freeze backbone
        'train_dl':    train_dl,
        'lr':          best_config['lr'],
        'wd':          best_config['wd'],
    },
]

ablation_results = []
print('Running ablation study (10 epochs each)...')
for cond in ablations:
    print(f"\n  {cond['name']}")
    val_acc, val_f1, history, _ = train_model(
        cond['model'], cond['train_dl'], val_dl,
        epochs=10, lr=cond['lr'], weight_decay=cond['wd'],
        verbose=False
    )
    ablation_results.append({
        'name': cond['name'],
        'val_acc': val_acc,
        'val_f1': val_f1,
        'history': history
    })

In [ ]:
# Display ablation results
baseline_acc = ablation_results[0]['val_acc']

print('Ablation study results:')
print('=' * 65)
print(f'  {'Condition':<30} {'Val Acc':>8} {'vs Baseline':>12}')
print('  ' + '-' * 52)
for r in ablation_results:
    delta = r['val_acc'] - baseline_acc
    sign  = '+' if delta >= 0 else ''
    print(f"  {r['name']:<30} {r['val_acc']:>8.3f} {sign+f'{delta:.3f}':>12}")
print('=' * 65)

# Plot accuracy curves
fig, ax = plt.subplots(figsize=(10, 5))
for r in ablation_results:
    ax.plot(range(1, 11), r['history']['val_acc'],
            label=r['name'], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation accuracy')
ax.set_title('Ablation study — effect of each component')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nReading the results:')
print('Components with a large negative delta matter a lot.')
print('Components with a small delta matter little — or could be removed.')

## 3. Logging results

As experiments accumulate, keeping track of what you tried and what 
worked becomes the main challenge. The minimal solution is a results 
log — a dictionary you save to disk after every experiment.

This is not as sophisticated as Weights & Biases, but it requires no 
setup and works reliably. Upgrade to a proper experiment tracker 
if you find yourself running more than ~20 experiments.

In [ ]:
class ExperimentLog:
    """
    Minimal experiment logger.
    Records hyperparameters and metrics for each experiment run,
    and saves to disk so results persist across Colab sessions.
    """
    def __init__(self, filepath='experiment_log.json'):
        self.filepath = filepath
        # Load existing log if it exists
        if Path(filepath).exists():
            with open(filepath) as f:
                self.runs = json.load(f)
        else:
            self.runs = []

    def log(self, name, config, metrics, notes=''):
        """
        Record one experiment run.

        Args:
            name:    short descriptive name for this run
            config:  dict of hyperparameters
            metrics: dict of evaluation metrics
            notes:   optional free-text notes
        """
        import datetime
        run = {
            'name':      name,
            'timestamp': datetime.datetime.now().isoformat(),
            'config':    config,
            'metrics':   metrics,
            'notes':     notes,
        }
        self.runs.append(run)
        self._save()
        print(f'Logged: {name} — {metrics}')

    def show(self):
        """Print a summary table of all logged runs."""
        if not self.runs:
            print('No runs logged yet.')
            return
        # Collect all metric keys
        all_metrics = sorted({k for r in self.runs for k in r['metrics']})
        header = f"  {'Name':<35}" + ''.join(f'{m:>10}' for m in all_metrics)
        print(header)
        print('  ' + '-' * len(header))
        for r in self.runs:
            row = f"  {r['name']:<35}"
            for m in all_metrics:
                val = r['metrics'].get(m, '-')
                row += f'{val:>10}' if isinstance(val, str) else f'{val:>10.3f}'
            print(row)

    def best(self, metric='val_acc'):
        """Return the run with the highest value of the given metric."""
        valid = [r for r in self.runs if metric in r['metrics']]
        return max(valid, key=lambda r: r['metrics'][metric]) if valid else None

    def _save(self):
        with open(self.filepath, 'w') as f:
            json.dump(self.runs, f, indent=2)


# Example usage
log = ExperimentLog('my_experiments.json')

# Log the ablation results
for r in ablation_results:
    log.log(
        name=r['name'],
        config={'epochs': 10, 'lr': best_config['lr'], 'wd': best_config['wd']},
        metrics={'val_acc': r['val_acc'], 'val_f1': r['val_f1']}
    )

print('\nAll logged runs:')
log.show()
print(f"\nBest run: {log.best('val_acc')['name']}")

## 4. Building custom architectures

Sometimes a pretrained ResNet with a replaced head is not enough. 
You might need:
- A custom classification head with more layers
- A model with multiple outputs (e.g. class label + bounding box)
- An encoder-decoder for segmentation or generation tasks

This section shows the most common patterns.

### Pattern 1: Custom classification head

The default head is a single `nn.Linear`. You can replace it with 
anything — a deeper MLP, a head with dropout, or one with batch norm.

In [ ]:
def make_custom_head_model(num_classes, hidden_dim=256, dropout=0.5):
    """
    ResNet-18 with a deeper classification head.
    Useful when the single linear layer is too simple for your task.
    """
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = backbone.fc.in_features   # 512 for ResNet-18

    # Replace the head with a deeper MLP
    backbone.fc = nn.Sequential(
        nn.Linear(in_features, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, num_classes)
    )
    return backbone


model = make_custom_head_model(NUM_CLASSES)
# Check the new head
print('Custom head:')
print(model.fc)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {trainable:,}')

### Pattern 2: Multi-output model

Some tasks require multiple outputs from the same backbone — for example, 
predicting a class label AND a bounding box (classification + localisation), 
or predicting age AND gender from a face image.

The pattern is to extract features from the backbone, then pass them 
through separate output heads.

In [ ]:
class MultiOutputModel(nn.Module):
    """
    Shared backbone with two separate output heads.

    Example use case: predict both the class label and a confidence score
    for each prediction (useful when you want calibrated probabilities).

    Adapt head_1 and head_2 for your specific task.
    """
    def __init__(self, num_classes):
        super().__init__()

        # Shared backbone — extracts features for both heads
        backbone    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = backbone.fc.in_features

        # Remove the original head
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        # backbone output: (batch, 512, 1, 1) — need to flatten

        # Head 1: class prediction
        self.head_class = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, num_classes)
        )

        # Head 2: bounding box prediction (x1, y1, x2, y2 normalised to [0, 1])
        # Replace with whatever your second task requires
        self.head_bbox = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()   # constrain output to [0, 1]
        )

    def forward(self, x):
        # Shared feature extraction
        features = self.backbone(x)          # (batch, 512, 1, 1)

        # Two independent outputs from the same features
        class_logits = self.head_class(features)  # (batch, num_classes)
        bbox_coords  = self.head_bbox(features)   # (batch, 4)

        return class_logits, bbox_coords


# Test the model
model = MultiOutputModel(NUM_CLASSES)
dummy_input = torch.randn(4, 3, 224, 224)
class_out, bbox_out = model(dummy_input)
print(f'Class logits shape: {class_out.shape}')  # (4, num_classes)
print(f'Bbox output shape:  {bbox_out.shape}')   # (4, 4)
print(f'Bbox values (should be in [0,1]): {bbox_out[0].detach().numpy().round(3)}')

# Training a multi-output model requires a combined loss:
# total_loss = cls_weight * cross_entropy(class_out, labels)
#            + box_weight * smooth_l1(bbox_out, target_boxes)

### Pattern 3: Encoder-decoder architecture

For segmentation, generation, or image-to-image tasks, you need an 
architecture that produces an output image rather than a single label. 
The encoder-decoder pattern is the standard approach:

- **Encoder:** compress the input into a compact representation 
  (use a pretrained backbone)
- **Decoder:** expand back to the output size 
  (upsampling layers + convolutions)
- **Skip connections:** pass feature maps from encoder to decoder 
  to preserve spatial detail (this is what makes U-Net work)

Here is a minimal encoder-decoder for semantic segmentation:

In [ ]:
class SimpleSegmentationModel(nn.Module):
    """
    Minimal encoder-decoder for semantic segmentation.

    Encoder: ResNet-18 backbone (pretrained)
    Decoder: progressive upsampling back to input resolution

    This is a simplified version — for production use, consider
    torchvision's DeepLabV3 or a library like segmentation-models-pytorch.
    """
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Encoder: use ResNet layers as feature extractors at different scales
        self.enc1 = nn.Sequential(backbone.conv1, backbone.bn1,
                                   backbone.relu, backbone.maxpool)  # /4
        self.enc2 = backbone.layer1   # /4,  64 channels
        self.enc3 = backbone.layer2   # /8,  128 channels
        self.enc4 = backbone.layer3   # /16, 256 channels
        self.enc5 = backbone.layer4   # /32, 512 channels

        # Decoder: upsample back to original resolution
        # Each block doubles the spatial size and halves the channels
        self.dec4 = self._decoder_block(512 + 256, 256)
        self.dec3 = self._decoder_block(256 + 128, 128)
        self.dec2 = self._decoder_block(128 + 64,  64)
        self.dec1 = self._decoder_block(64,         32)

        # Final 1x1 conv to get per-pixel class scores
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def _decoder_block(self, in_channels, out_channels):
        """Upsample by 2x then apply two convolutions."""
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        input_size = x.shape[2:]

        # Encoder — save intermediate features for skip connections
        e1 = self.enc1(x)   # (B,  64, H/4,  W/4)
        e2 = self.enc2(e1)  # (B,  64, H/4,  W/4)
        e3 = self.enc3(e2)  # (B, 128, H/8,  W/8)
        e4 = self.enc4(e3)  # (B, 256, H/16, W/16)
        e5 = self.enc5(e4)  # (B, 512, H/32, W/32)

        # Decoder — concatenate skip connections at each scale
        d4 = self.dec4(torch.cat([e5, e4], dim=1))  # skip from enc4
        d3 = self.dec3(torch.cat([d4, e3], dim=1))  # skip from enc3
        d2 = self.dec2(torch.cat([d3, e2], dim=1))  # skip from enc2
        d1 = self.dec1(d2)

        # Final prediction — upsample to input resolution
        out = self.head(d1)
        out = nn.functional.interpolate(out, size=input_size,
                                         mode='bilinear', align_corners=False)
        return out   # (B, num_classes, H, W)


# Test the segmentation model
seg_model   = SimpleSegmentationModel(num_classes=21)  # e.g. Pascal VOC
dummy_input = torch.randn(2, 3, 224, 224)
output      = seg_model(dummy_input)
print(f'Input shape:  {dummy_input.shape}')   # (2, 3, 224, 224)
print(f'Output shape: {output.shape}')        # (2, 21, 224, 224) — one score per pixel per class

params = sum(p.numel() for p in seg_model.parameters())
print(f'Total parameters: {params:,}')

## 5. Your experiment plan for today

For the rest of the session, apply what you have learned to your project.

**Step 1 — Set up your experiment log.**
Create an `ExperimentLog` at the start of your project notebook. 
Log your baseline result as run 0.

**Step 2 — Run a learning rate search.**
Use the coarse search approach from section 1. 
5 epochs on 30% of your data takes only a few minutes.

**Step 3 — Design one ablation.**
Pick one component of your model or training setup and remove or 
change it. What do you expect to happen? Run it and find out.

**Step 4 — Consider whether you need a custom architecture.**
Look at the patterns in section 4. Does your task fit standard classification, 
or do you need multiple outputs, an encoder-decoder, or a custom head? 
If you are unsure, ask a TA.

**Questions to be able to answer before you leave:**
- What is your best learning rate?
- What is your best result so far, and how does it compare to your baseline?
- What is the next experiment you will run, and why?
- Do you need a custom architecture, and if so which pattern applies?